In [1]:
import pandas as pd
import numpy as np
import faiss
import ollama
import json
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Load your dataset
df = pd.read_csv('../dataset/books_genre.csv')

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSample row:")
print(df.iloc[0][['book_id','title','authors','tag_name','content','content_features','average_rating']].to_dict())

Shape: (9818, 27)

Columns: ['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year', 'original_title', 'title', 'language_code', 'average_rating', 'ratings_count', 'work_ratings_count', 'work_text_reviews_count', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url', 'small_image_url', 'content', 'goodreads_book_id', 'tag_name', 'content_features']

Sample row:
{'book_id': 2767052, 'title': 'the hunger games the hunger games 1', 'authors': 'Suzanne Collins', 'tag_name': 'favorites currently-reading young-adult fiction dystopian to-read dystopia fantasy ya science-fiction books-i-own sci-fi series owned favourites romance adventure hunger-games book-club kindle teen read-in-2012 post-apocalyptic my-books the-hunger-games favorite-books action suzanne-collins re-read all-time-favorites ya-fiction survival sci-fi-fantasy books favorite scifi 5-stars i-own read-in-2011 novels ebook audiobook young-adult-fic

In [2]:
# See what content actually looks like
for col in ['content', 'content_features', 'tag_name']:
    sample = df[col].dropna().iloc[0] if col in df.columns else "MISSING"
    coverage = df[col].notna().sum() if col in df.columns else 0
    print(f"\n=== {col} ===")
    print(f"Coverage: {coverage}/{len(df)} ({coverage/len(df)*100:.1f}%)")
    print(f"Sample: {str(sample)[:300]}")


=== content ===
Coverage: 9818/9818 (100.0%)
Sample: the hunger games the hunger games 1 Suzanne Collins 2008

=== content_features ===
Coverage: 9803/9818 (99.8%)
Sample: the hunger games the hunger games 1 Suzanne Collins favorites currently-reading young-adult fiction dystopian to-read dystopia fantasy ya science-fiction books-i-own sci-fi series owned favourites romance adventure hunger-games book-club kindle teen read-in-2012 post-apocalyptic my-books the-hunger-

=== tag_name ===
Coverage: 9818/9818 (100.0%)
Sample: favorites currently-reading young-adult fiction dystopian to-read dystopia fantasy ya science-fiction books-i-own sci-fi series owned favourites romance adventure hunger-games book-club kindle teen read-in-2012 post-apocalyptic my-books the-hunger-games favorite-books action suzanne-collins re-read 


In [3]:
def build_book_text(row):
    """
    Combine all text signals into one string for embedding.
    nomic-embed-text handles up to 8192 tokens — be generous.
    """
    parts = []
    
    # Title + author (always include)
    if pd.notna(row.get('title')):
        parts.append(f"Book: {row['title']}")
    if pd.notna(row.get('authors')):
        parts.append(f"by {row['authors']}")
    
    # Tags — most valuable for mood matching
    if pd.notna(row.get('tag_name')) and str(row['tag_name']).strip():
        parts.append(f"Genre: {row['tag_name']}")
    
    # Content features — Goodreads tag cloud, very rich
    if pd.notna(row.get('content')) and str(row['content']).strip():
        content_str = str(row['content']).strip()
        # Replace underscores/hyphens with spaces for better embedding
        content_str = content_str.replace('-', ' ').replace('_', ' ')
        parts.append(f"Themes: {content_str[:600]}")
    
    if pd.notna(row.get('content_features')) and str(row['content_features']).strip():
        cf = str(row['content_features']).strip()
        # Only add if it's actually text, not all numbers
        if any(c.isalpha() for c in cf):
            parts.append(f"Features: {cf[:300]}")
    
    return ' | '.join(parts)

df['book_text'] = df.apply(build_book_text, axis=1)

# Quality check
print("Sample book_text entries:")
for i in [0, 100, 500, 1000]:
    if i < len(df):
        print(f"\n--- Row {i} ---")
        print(df['book_text'].iloc[i][:400])

print(f"\nAvg text length: {df['book_text'].str.len().mean():.0f} chars")
print(f"Empty texts: {(df['book_text'].str.len() < 10).sum()}")

Sample book_text entries:

--- Row 0 ---
Book: the hunger games the hunger games 1 | by Suzanne Collins | Genre: favorites currently-reading young-adult fiction dystopian to-read dystopia fantasy ya science-fiction books-i-own sci-fi series owned favourites romance adventure hunger-games book-club kindle teen read-in-2012 post-apocalyptic my-books the-hunger-games favorite-books action suzanne-collins re-read all-time-favorites ya-fictio

--- Row 100 ---
Book: the count of monte cristo | by Alexandre Dumas, Robin Buss | Genre: classics favorites fiction to-read classic historical-fiction adventure owned books-i-own french historical clàssics favourites france kindle classic-literature novels romance to-buy classics-to-read 19th-century my-ebooks french-literature owned-books 1001-books book-club library ebook to-read-classics default audiobook all

--- Row 500 ---
Book: the hiding place the triumphant true story of corrie ten boom | by Corrie ten Boom, John Sherrill, Elizabeth Sherrill

In [4]:
def embed_texts_ollama(texts: list, batch_size: int = 50) -> np.ndarray:
    """
    Use Ollama nomic-embed-text for embeddings.
    nomic-embed-text produces 768-dim vectors.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i : i + batch_size]
        batch_embeddings = []
        
        for text in batch:
            try:
                response = ollama.embed(
                    model='nomic-embed-text',
                    input=text
                )
                # ollama returns {'embeddings': [[...vector...]]}
                emb = response['embeddings'][0]
                batch_embeddings.append(emb)
            except Exception as e:
                print(f"Error embedding text: {e}")
                # Use zero vector as fallback
                batch_embeddings.append([0.0] * 768)
        
        all_embeddings.extend(batch_embeddings)
    
    return np.array(all_embeddings, dtype=np.float32)


# This will take 10-30 min depending on dataset size
# nomic-embed-text is fast locally
texts = df['book_text'].tolist()
print(f"Embedding {len(texts)} books...")

embeddings = embed_texts_ollama(texts, batch_size=50)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
# Expected: (n_books, 768)

Embedding 9818 books...


Embedding: 100%|██████████| 197/197 [19:30<00:00,  5.94s/it]



Embedding matrix shape: (9818, 768)


In [5]:
import os
os.makedirs('models', exist_ok=True)

# L2 normalize for cosine similarity via inner product
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1, norms)  # avoid div by zero
embeddings_norm = (embeddings / norms).astype(np.float32)

# Save
np.save('models/book_embeddings.npy', embeddings_norm)
print(f"Saved embeddings: {embeddings_norm.shape}")

# Save book metadata for lookup (index position → book info)
meta_cols = ['book_id', 'title', 'authors', 'average_rating', 
             'ratings_count', 'tag_name', 'image_url']
meta_cols = [c for c in meta_cols if c in df.columns]  # only existing cols

df_meta = df[meta_cols].reset_index(drop=True)
df_meta.to_pickle('models/book_meta.pkl')
print(f"Saved metadata: {df_meta.shape}")
print(df_meta.head(3))

Saved embeddings: (9818, 768)
Saved metadata: (9818, 7)
   book_id                                              title  \
0  2767052                the hunger games the hunger games 1   
1        3  harry potter and the sorcerers stone harry pot...   
2    41865                                twilight twilight 1   

                       authors  average_rating  ratings_count  \
0              Suzanne Collins            4.34        4780653   
1  J.K. Rowling, Mary GrandPré            4.44        4602479   
2              Stephenie Meyer            3.57        3866839   

                                            tag_name  \
0  favorites currently-reading young-adult fictio...   
1  to-read favorites fantasy currently-reading yo...   
2  young-adult fantasy favorites vampires ya fict...   

                                           image_url  
0  https://images.gr-assets.com/books/1447303603m...  
1  https://images.gr-assets.com/books/1474154022m...  
2  https://images.gr-assets.com/

In [6]:
dim = embeddings_norm.shape[1]  # 768 for nomic-embed-text
print(f"Building FAISS index with dim={dim}, n_books={len(embeddings_norm)}")

# IndexFlatIP = exact inner product search (= cosine sim on normalized vecs)
# Best for < 100k books. Your goodbooks dataset is ~10k so this is perfect.
index = faiss.IndexFlatIP(dim)
index.add(embeddings_norm)

print(f"Index contains {index.ntotal} vectors")

faiss.write_index(index, 'models/book_faiss.index')
print("FAISS index saved.")

Building FAISS index with dim=768, n_books=9818
Index contains 9818 vectors
FAISS index saved.


In [7]:
def query_embed(text: str) -> np.ndarray:
    """Embed a single query string."""
    response = ollama.embed(model='nomic-embed-text', input=text)
    vec = np.array(response['embeddings'][0], dtype=np.float32)
    vec = vec / np.linalg.norm(vec)
    return vec.reshape(1, -1)

def semantic_search(query: str, top_n: int = 10) -> pd.DataFrame:
    vec = query_embed(query)
    scores, indices = index.search(vec, top_n)
    
    results = df_meta.iloc[indices[0]].copy()
    results['semantic_score'] = scores[0]
    return results[['title', 'authors', 'tag_name', 'average_rating', 'semantic_score']]


# ── Test various moods ──────────────────────────────────────────

print("=== HAPPY / UPLIFTING ===")
print(semantic_search("uplifting joyful feel-good heartwarming optimistic happy story", 8))

print("\n=== SAD / EMOTIONAL ===")
print(semantic_search("emotional grief melancholic loss sadness cathartic quiet", 8))

print("\n=== RAINY COSY DAY ===")
print(semantic_search("cosy atmospheric moody rainy day introspective stay indoors", 8))

print("\n=== SUMMER BEACH ===")
print(semantic_search("light fun beach holiday summer adventure fast paced escapism", 8))

print("\n=== LATE NIGHT / CANT SLEEP ===")
print(semantic_search("gripping dark suspense late night page turner twists thriller", 8))

print("\n=== TRAVEL LONG FLIGHT ===")
print(semantic_search("epic immersive long journey fantasy world building adventure", 8))

=== HAPPY / UPLIFTING ===
                                                  title  \
7806                                  happy happy happy   
4904                  feeling good the new mood therapy   
2908  this is what happy looks like this is what hap...   
4868                           the pursuit of happyness   
946   the happiness project or why i spent a year tr...   
6862              this is the story of a happy marriage   
1471                     the power of positive thinking   
7205  learned optimism how to change your mind and y...   

                             authors  \
7806  Phil Robertson, Mark Schlabach   
4904                  David D. Burns   
2908               Jennifer E. Smith   
4868                   Chris Gardner   
946                   Gretchen Rubin   
6862                    Ann Patchett   
1471            Norman Vincent Peale   
7205            Martin E.P. Seligman   

                                               tag_name  average_rating  \
7806  

In [8]:
def expand_query_with_llm(
    mood: str,
    context: dict = None
) -> str:
  
    ctx_lines = []
    if context:
        if context.get('season'):     ctx_lines.append(f"Season: {context['season']}")
        if context.get('time_of_day'):ctx_lines.append(f"Time: {context['time_of_day']}")
        if context.get('travel'):     ctx_lines.append(f"Travel type: {context['travel']}")
        if context.get('user_genres'):ctx_lines.append(f"User enjoys: {', '.join(context['user_genres'])}")
        if context.get('reading_time'):ctx_lines.append(f"Available reading time: {context['reading_time']}")
    
    ctx_str = '\n'.join(ctx_lines) if ctx_lines else 'None'
    
    prompt = f"""You are a book recommendation assistant helping find the perfect book.

Convert the reading mood and context below into a rich 2-3 sentence description 
of what kind of book would perfectly match. Focus on themes, emotions, pacing, 
atmosphere, and narrative style — NOT just genre names.

Output ONLY the description. No intro, no labels, no bullet points.

Mood: {mood}
Additional context:
{ctx_str}

Book description:"""

    response = ollama.chat(
        model='llama3.2:1b',
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response['message']['content'].strip()


# Test combinations
test_cases = [
    {
        'mood': 'happy and energetic',
        'context': {'season': 'summer', 'time_of_day': 'afternoon', 'user_genres': ['thriller', 'fantasy']}
    },
    {
        'mood': 'sad and reflective',
        'context': {'season': 'winter', 'time_of_day': 'night'}
    },
    {
        'mood': 'want something cosy',
        'context': {'season': 'rainy', 'reading_time': '30 minutes'}
    },
    {
        'mood': 'adventurous and excited',
        'context': {'travel': 'long flight', 'user_genres': ['sci-fi', 'fantasy']}
    },
]

for tc in test_cases:
    expanded = expand_query_with_llm(tc['mood'], tc.get('context', {}))
    print(f"\nMood: {tc['mood']}")
    print(f"Context: {tc.get('context', {})}")
    print(f"Expanded → {expanded}")
    print(f"\nTop results:")
    print(semantic_search(expanded, 5)[['title', 'authors', 'semantic_score']])
    print("─" * 60)


Mood: happy and energetic
Context: {'season': 'summer', 'time_of_day': 'afternoon', 'user_genres': ['thriller', 'fantasy']}
Expanded → The sun beats down on the lush green lawn as you lounge in your favorite outdoor chair, surrounded by blooming flowers that seem to dance in the gentle breeze. The air is filled with the sweet scent of sizzling burgers and the sound of laughter carries through the afternoon heat. As the warmth spreads through your skin, you feel invigorated and ready to take on whatever the day may bring.

In this idyllic scene, a sense of freedom and abandon washes over you, and you're drawn into a world of excitement and danger as a mysterious figure appears out of nowhere, leaving a trail of clues that lead you on a thrilling adventure. Your heart beats faster with every new discovery, and your senses come alive as the story unfolds with unexpected twists and turns.

The pace is quick-witted and engaging, keeping you on the edge of your seat as the stakes grow highe

In [12]:
# Load your existing models
import joblib

cosine_sim = joblib.load('../models/cosine_sim.pkl')
# with open('../models/cosine_sim.pkl', 'rb') as f:
#     cosine_sim = pickle.load(f)

# Get title→index mapping for cosine lookup
title_to_idx = pd.Series(df.index, index=df['title'].str.lower()).drop_duplicates()

def get_cosine_scores_nb(seed_title: str, top_n: int = 100) -> dict:
    """Returns {title: cosine_score} for top_n similar books."""
    seed_lower = seed_title.lower()
    if seed_lower not in title_to_idx:
        return {}
    idx = title_to_idx[seed_lower]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    return {df['title'].iloc[i]: float(s) for i, s in sim_scores}

def fuse_scores(
    semantic_query: str,
    seed_title: str = None,
    semantic_w: float = 0.5,
    content_w: float = 0.3,
    rating_w: float = 0.2,
    top_n: int = 12
) -> pd.DataFrame:
    
    # 1. Semantic scores
    vec = query_embed(semantic_query)
    sem_scores_raw, sem_indices = index.search(vec, 200)
    sem_df = df_meta.iloc[sem_indices[0]].copy()
    sem_df['sem'] = sem_scores_raw[0]
    # Normalize 0-1
    sem_range = sem_df['sem'].max() - sem_df['sem'].min()
    if sem_range > 0:
        sem_df['sem_norm'] = (sem_df['sem'] - sem_df['sem'].min()) / sem_range
    else:
        sem_df['sem_norm'] = 1.0
    
    # 2. Content/cosine scores
    content_scores = {}
    if seed_title:
        content_scores = get_cosine_scores_nb(seed_title, top_n=200)
    
    # 3. Rating popularity boost (log-normalized)
    max_rc = df_meta['ratings_count'].max() if 'ratings_count' in df_meta.columns else 1
    
    # 4. Fuse
    fused_rows = []
    for _, row in sem_df.iterrows():
        title = row['title']
        sem_n  = row['sem_norm']
        cont_n = content_scores.get(title, 0.0)
        
        rc = row.get('ratings_count', 0) or 0
        rat_n  = np.log1p(rc) / np.log1p(max_rc) if max_rc > 0 else 0
        
        final  = semantic_w * sem_n + content_w * cont_n + rating_w * rat_n
        fused_rows.append({
            'title': title,
            'authors': row.get('authors', ''),
            'tag_name': row.get('tag_name', ''),
            'average_rating': row.get('average_rating', 0),
            'sem_score': round(sem_n, 3),
            'content_score': round(cont_n, 3),
            'rating_boost': round(rat_n, 3),
            'final_score': round(final, 3),
        })
    
    result = pd.DataFrame(fused_rows)
    result = result.sort_values('final_score', ascending=False).head(top_n)
    return result.reset_index(drop=True)

# Test
expanded = expand_query_with_llm("thriller murder mystery book read", {'season': 'rainy'})
print(f"Expanded query: {expanded}\n")
results = fuse_scores(expanded, seed_title=None, top_n=10)
print(results[['title', 'authors', 'sem_score', 'content_score', 'final_score']])

Expanded query: The story unfolds with a sense of claustrophobia, as the protagonist is trapped in a small, isolated space where every creaking floorboard and dripping raindrop feels like a potential killer's sinister move. The atmosphere is heavy with tension, as the reader is kept on edge by the eerie silence and the unsettling feeling that someone (or something) is lurking just out of sight. As the investigation unfolds, the pacing quickens, but only to reveal more secrets and lies beneath the surface, leaving the reader questioning everything they thought they knew about the crime and its perpetrator.

                     title            authors  sem_score  content_score  \
0    the weight of silence  Heather Gudenkauf      1.000            0.0   
1      the weight of blood       Laura McHugh      0.823            0.0   
2              sleep tight      Rachel Abbott      0.809            0.0   
3      behind closed doors         B.A. Paris      0.757            0.0   
4          